## **Feature Engineering**

### **Import Libraries and Load Dataset**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

daily_df = pd.read_csv(os.path.join(PROCESSED_DIR, "store_dept_daily_sales.csv"))
daily_df['date'] = pd.to_datetime(daily_df['date'])

print("Dataset shape:", daily_df.shape)
display(daily_df.head())


Dataset shape: (5739, 14)


,date,dept_id,weekday,month,year,snap_CA,is_event,event_count,is_cultural_event,is_national_event,is_religious_event,is_sport_event,sales_qty,sell_price
0,2011-01-29,FOODS_1,Saturday,1,2011,0,0,0,0,0,0,0,341,2.976216
1,2011-01-30,FOODS_1,Sunday,1,2011,0,0,0,0,0,0,0,326,2.976216
2,2011-01-31,FOODS_1,Monday,1,2011,0,0,0,0,0,0,0,260,2.976216
3,2011-02-01,FOODS_1,Tuesday,2,2011,1,0,0,0,0,0,0,231,2.976216
4,2011-02-02,FOODS_1,Wednesday,2,2011,1,0,0,0,0,0,0,220,2.976216


### **Sort Dataset for Time-Series Feature Creation**

The dataset must be sorted by department and date before any lag or rolling feature is created. This ensures that when values are shifted to create a lag, the operation always looks backwards in time within the correct department.

In [2]:
daily_df = daily_df.sort_values(['dept_id','date']).reset_index(drop=True)
print("Departments:", sorted(daily_df['dept_id'].unique().tolist()))
print("Date range:", daily_df['date'].min().date(), "to", daily_df['date'].max().date())


Departments: ['FOODS_1', 'FOODS_2', 'FOODS_3']
Date range: 2011-01-29 to 2016-04-24


### **Lag Features**

**What they are:** A lag feature copies the sales value from N days in the past into a new column on the current row. For example, `lag_7` on a Wednesday holds the sales from the previous Wednesday.

**Why they are needed:** Tree-based ML models like XGBoost and LightGBM work on tabular rows — they cannot read time sequences the way ARIMA can. Lag features are the way to give the model access to historical demand context. As a demand planner, this is similar to the manual practice of writing last week's sales in a column next to this week's order plan.

**Leakage prevention:** All lags use `.shift(n)`, which moves values forward in time. This means every lag column on a given row only contains data from before that row's date — the model never sees future data.

| Lag | Signal captured |
|-----|----------------|
| lag_1, 2, 3 | Last 3 days — immediate recent demand |
| lag_7 | Same weekday last week — weekly cycle |
| lag_14, 21 | Two and three weeks ago |
| lag_28 | Same period last month |
| lag_42, 56, 84 | Medium to long-term demand memory |


In [3]:
for lag in [1,2,3,7,14,21,28,42,56,84]:
    daily_df[f'lag_{lag}'] = daily_df.groupby('dept_id')['sales_qty'].shift(lag)

print("Lag features created:", [c for c in daily_df.columns if c.startswith('lag_')])


Lag features created: ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_42', 'lag_56', 'lag_84']


### **Rolling Mean and Standard Deviation Features**

**What they are:** A rolling mean is the average of sales over the past N days. A rolling standard deviation measures how much demand varied over the same window.

**Why they are needed:** A single lag captures one past day. A rolling mean gives the model a picture of the recent demand level across a wider window, smoothing out day-to-day noise. The rolling standard deviation adds a volatility signal — high std means demand has been erratic, which a demand planner would factor into safety stock.

**Shift-before-rolling:** The `.shift(1).rolling(N)` pattern ensures that today's actual value is not included in the window, preventing the model from using today's real sales as a feature to predict today's sales.

Windows used: **7, 14, 28, 42, 56 days**


In [4]:
for window in [7,14,28,42,56]:
    daily_df[f'rolling_mean_{window}'] = (
        daily_df.groupby('dept_id')['sales_qty']
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )
for window in [7,14,28,42,56]:
    daily_df[f'rolling_std_{window}'] = (
        daily_df.groupby('dept_id')['sales_qty']
        .transform(lambda x: x.shift(1).rolling(window).std())
    )

print("Rolling mean:", [c for c in daily_df.columns if 'rolling_mean' in c])
print("Rolling std: ", [c for c in daily_df.columns if 'rolling_std'  in c])


Rolling mean: ['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_mean_42', 'rolling_mean_56']
Rolling std:  ['rolling_std_7', 'rolling_std_14', 'rolling_std_28', 'rolling_std_42', 'rolling_std_56']


### **Exponentially Weighted Moving Average (EWMA) Features**

**What they are:** EWMA is a weighted average that gives more weight to recent days and less weight to older days, with the weights decreasing exponentially the further back in time.

**Why they are needed:** In demand planning, what happened last week is more relevant than what happened two months ago. Standard rolling means give equal weight to all days in the window. EWMA captures the recency effect more accurately. It also reacts faster to recent demand shifts, which is useful when trying to detect trend changes early.

Spans used: **7, 14, 28 days**

In [5]:
for span in [7,14,28]:
    daily_df[f'ewma_{span}'] = (
        daily_df.groupby('dept_id')['sales_qty']
        .transform(lambda x: x.shift(1).ewm(span=span).mean())
    )

print("EWMA features:", [c for c in daily_df.columns if c.startswith('ewma_')])


EWMA features: ['ewma_7', 'ewma_14', 'ewma_28']


### **Rolling Maximum and Minimum Features**

**What they are:** Rolling max captures the highest sales value seen over the past N days. Rolling min captures the lowest.

**Why they are needed:** Rolling means and EWMA smooth out demand spikes. Rolling max preserves the spike signal — telling the model that demand reached a certain peak level recently. As a demand planner, tracking peak daily demand is directly relevant to setting safety stock. Rolling min captures the demand floor, useful for identifying slow periods and planning minimum order quantities.

Windows used: **7 days** and **28 days**


In [6]:
for window in [7,28]:
    daily_df[f'rolling_max_{window}'] = (
        daily_df.groupby('dept_id')['sales_qty']
        .transform(lambda x: x.shift(1).rolling(window).max())
    )
    daily_df[f'rolling_min_{window}'] = (
        daily_df.groupby('dept_id')['sales_qty']
        .transform(lambda x: x.shift(1).rolling(window).min())
    )

print("Rolling max:", [c for c in daily_df.columns if 'rolling_max' in c])
print("Rolling min:", [c for c in daily_df.columns if 'rolling_min' in c])


Rolling max: ['rolling_max_7', 'rolling_max_28']
Rolling min: ['rolling_min_7', 'rolling_min_28']


### **Calendar Features**

**What they are:** Features derived from the date — day of the week, day of the month, whether it falls on a weekend, and whether it is the last day of a month or quarter.

**Why they are needed:** Consumer behavior in grocery retail follows strong calendar rhythms. Weekends typically see higher foot traffic. End-of-month demand often rises because many households receive salary or SNAP benefits near month-end. Quarter-end can trigger elevated purchasing in some food categories.

**Note on weekday overlap:** The original dataset from preprocessing already contains a `weekday` column (sourced from the calendar file). `day_of_week` created below contains the same values using pandas' `dt.dayofweek` accessor. Both are retained because `weekday` is part of the aggregation key in Step 3 while `day_of_week` is the clean numeric version used in ML features. The model will see them as correlated features, and those with low importance will be excluded in the feature selection step.

In [7]:
daily_df['day_of_week']    = daily_df['date'].dt.dayofweek
daily_df['week_of_year']   = daily_df['date'].dt.isocalendar().week.astype(int)
daily_df['is_weekend']     = daily_df['day_of_week'].isin([5,6]).astype(int)
daily_df['day_of_month']   = daily_df['date'].dt.day
daily_df['is_month_end']   = daily_df['date'].dt.is_month_end.astype(int)
daily_df['is_quarter_end'] = daily_df['date'].dt.is_quarter_end.astype(int)

print("Calendar features:")
for col in ['day_of_week','week_of_year','is_weekend','day_of_month','is_month_end','is_quarter_end']:
    print(f"  {col}: {daily_df[col].nunique()} unique values")


Calendar features:
  day_of_week: 7 unique values
  week_of_year: 53 unique values
  is_weekend: 2 unique values
  day_of_month: 31 unique values
  is_month_end: 2 unique values
  is_quarter_end: 2 unique values


### **Price-Based Features**

The average selling price at the department level is already available from the preprocessing step. The additional features below help the model detect whether a price change has recently occurred and how the price has moved relative to recent history.

The percentage change over 7 and 28 days captures the direction and magnitude of recent price movement. Absolute price alone does not tell the model whether a price just went up or down.

The price lag features (`sell_price_lag_7`, `sell_price_lag_28`) give the model access to recent price history, which helps detect delayed consumer responses — shoppers may react to a price change over the following days or weeks rather than immediately.


In [8]:
daily_df['price_change_7']    = daily_df.groupby('dept_id')['sell_price'].pct_change(7)
daily_df['price_change_28']   = daily_df.groupby('dept_id')['sell_price'].pct_change(28)
daily_df['sell_price_lag_7']  = daily_df.groupby('dept_id')['sell_price'].shift(7)
daily_df['sell_price_lag_28'] = daily_df.groupby('dept_id')['sell_price'].shift(28)

print("Price features:")
print(daily_df[['price_change_7','price_change_28','sell_price_lag_7','sell_price_lag_28']].describe())


Price features:
       price_change_7  price_change_28  sell_price_lag_7  sell_price_lag_28
count     5718.000000      5655.000000       5718.000000        5655.000000
mean         0.000393         0.001478          3.379629           3.378541
std          0.004229         0.008970          0.515954           0.515539
min         -0.015887        -0.025544          2.725660           2.725660
25%         -0.001455        -0.003106          2.853696           2.851634
50%          0.000000         0.000518          3.288657           3.287546
75%          0.001484         0.005182          4.003354           4.000465
max          0.052577         0.069083          4.223829           4.223829


### **Trend Index**

A simple counter that starts at 0 for the first row of a department and increases by 1 for each subsequent day.

Tree-based models treat each row independently — they have no built-in awareness that one row comes after another. The trend index gives the model an explicit time position, allowing it to learn whether overall demand has been gradually increasing or declining across the full history.

In [9]:
daily_df['trend_index'] = daily_df.groupby('dept_id').cumcount()
print("Trend index range per department:")
print(daily_df.groupby('dept_id')['trend_index'].agg(['min','max']))


Trend index range per department:
         min   max
dept_id           
FOODS_1    0  1912
FOODS_2    0  1912
FOODS_3    0  1912


### **SNAP x Event Interaction Feature**

SNAP benefit days and special events each independently drive food demand. However, when both occur on the same day — for example, a national holiday that falls on a SNAP issuance date — the combined effect on demand may be greater than either factor alone.

This interaction term is 1 only when both `snap_CA = 1` and `is_event = 1` on the same day. It allows the model to learn from these "double driver" days separately from days when only one factor is active.

***SNAP or Supplemental Nutrition Assistance Program*** — it is the US federal government's food assistance program, commonly known as food stamps, that helps low-income households buy groceries.


In [10]:
daily_df['snap_event_inter'] = daily_df['snap_CA'] * daily_df['is_event']

print("SNAP x Event interaction value counts:")
print(daily_df['snap_event_inter'].value_counts().sort_index())
print("\nDays where both SNAP and an event occur:", (daily_df['snap_event_inter']==1).sum())


SNAP x Event interaction value counts:
snap_event_inter
0    5583
1     156
Name: count, dtype: int64

Days where both SNAP and an event occur: 156


### **Event Features (Confirmed from Preprocessing)**

The event indicator columns created in Step 2 are carried through and confirmed present in the dataset.

In [11]:
event_cols = ['is_event','event_count','is_cultural_event','is_national_event','is_religious_event','is_sport_event']
print("Event features confirmed present:", event_cols)
display(daily_df[event_cols].describe())


Event features confirmed present: ['is_event', 'event_count', 'is_cultural_event', 'is_national_event', 'is_religious_event', 'is_sport_event']


,is_event,event_count,is_cultural_event,is_national_event,is_religious_event,is_sport_event
count,5739.000000,5739.000000,5739.000000,5739.000000,5739.000000,5739.000000
mean,0.080502,0.082593,0.019864,0.026660,0.027705,0.008364
std,0.272092,0.282784,0.139545,0.161101,0.164141,0.091079
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000


### **Remove Rows with Missing Values**

Lag and rolling features produce NaN values at the beginning of each department series because there is no history before the first record. The longest lag used is `lag_84`, so the first 84 rows of each department will have at least one missing value.

These incomplete rows are removed rather than imputed. Filling lag values with estimated numbers would introduce artificial demand signals that never actually occurred, which would mislead the model during training.

After this step, each department's usable series starts from approximately day 85 of the original data, which still provides several years of complete daily records for training.

In [12]:
daily_df = daily_df.dropna().reset_index(drop=True)

print("Final modeling dataset shape:", daily_df.shape)
print("\nRows per department:")
print(daily_df.groupby('dept_id').size())
display(daily_df.head())


Final modeling dataset shape: (5487, 53)

Rows per department:
dept_id
FOODS_1    1829
FOODS_2    1829
FOODS_3    1829
dtype: int64


,date,dept_id,weekday,month,year,snap_CA,is_event,event_count,is_cultural_event,is_national_event,is_religious_event,is_sport_event,sales_qty,sell_price,lag_1,lag_2,lag_3,lag_7,lag_14,lag_21,lag_28,lag_42,lag_56,lag_84,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_mean_42,rolling_mean_56,rolling_std_7,rolling_std_14,rolling_std_28,rolling_std_42,rolling_std_56,ewma_7,ewma_14,ewma_28,rolling_max_7,rolling_min_7,rolling_max_28,rolling_min_28,day_of_week,week_of_year,is_weekend,day_of_month,is_month_end,is_quarter_end,price_change_7,price_change_28,sell_price_lag_7,sell_price_lag_28,trend_index,snap_event_inter
0,2011-04-23,FOODS_1,Saturday,4,2011,0,0,0,0,0,0,0,487,2.984344,303.0,274.0,204.0,320.0,278.0,329.0,353.0,332.0,463.0,341.0,281.857143,266.000000,265.285714,267.500000,273.196429,43.517922,57.412140,58.334830,53.892961,59.675614,272.025185,267.645001,267.089633,334.0,204.0,381.0,166.0,5,16,1,23,0,0,-0.006709,-0.00385,3.0045,2.995877,84,0
1,2011-04-24,FOODS_1,Sunday,4,2011,0,1,2,1,0,1,0,464,2.984344,487.0,303.0,274.0,334.0,381.0,373.0,340.0,322.0,349.0,326.0,305.714286,280.928571,270.071429,271.190476,273.625000,89.449640,82.474671,70.106231,62.961559,61.131918,325.768888,296.892486,282.290857,487.0,204.0,487.0,166.0,6,16,1,24,0,0,-0.006709,-0.00385,3.0045,2.995877,85,0
2,2011-04-25,FOODS_1,Monday,4,2011,0,0,0,0,0,0,0,243,2.984344,464.0,487.0,303.0,257.0,234.0,262.0,265.0,274.0,261.0,260.0,324.285714,286.857143,274.500000,274.571429,275.678571,107.894569,92.584977,78.143031,69.254618,65.486461,360.326666,319.173589,294.849438,487.0,204.0,487.0,166.0,0,17,0,25,0,0,-0.006709,-0.00385,3.0045,2.995877,86,0
3,2011-04-26,FOODS_1,Tuesday,4,2011,0,1,1,0,0,1,0,222,2.984344,243.0,464.0,487.0,281.0,166.0,219.0,217.0,266.0,262.0,231.0,322.285714,287.500000,273.714286,273.833333,275.357143,109.467977,92.220263,78.352412,69.425839,65.603888,330.995000,309.017071,291.266465,487.0,204.0,487.0,166.0,1,17,0,26,0,0,-0.006709,-0.00385,3.0045,2.995877,87,0
4,2011-04-27,FOODS_1,Wednesday,4,2011,0,0,0,0,0,0,0,190,2.984344,222.0,243.0,464.0,204.0,199.0,256.0,186.0,225.0,264.0,220.0,313.857143,291.500000,273.892857,272.785714,274.642857,115.293001,87.645922,78.223961,69.877431,65.968706,303.746250,297.414755,286.480576,487.0,204.0,487.0,166.0,2,17,0,27,0,0,-0.006709,-0.00385,3.0045,2.995877,88,0


### **Final Feature Summary**

The complete dataset ready for modeling contains the following columns. The total number of predictor features available to ML models is shown below.

**Target variable**
- `sales_qty`

**Original inputs from preprocessing (11 columns)**
- `weekday`, `month`, `year`, `snap_CA`, `sell_price`
- `is_event`, `event_count`, `is_cultural_event`, `is_national_event`,
  `is_religious_event`, `is_sport_event`

**Engineered features (39 columns)**

| Group | Features |
|-------|---------|
| Lag (10) | lag_1, lag_2, lag_3, lag_7, lag_14, lag_21, lag_28, lag_42, lag_56, lag_84 |
| Rolling mean (5) | rolling_mean_7/14/28/42/56 |
| Rolling std (5) | rolling_std_7/14/28/42/56 |
| Rolling max (2) | rolling_max_7/28 |
| Rolling min (2) | rolling_min_7/28 |
| EWMA (3) | ewma_7, ewma_14, ewma_28 |
| Calendar (6) | day_of_week, week_of_year, is_weekend, day_of_month, is_month_end, is_quarter_end |
| Price (4) | price_change_7, price_change_28, sell_price_lag_7, sell_price_lag_28 |
| Trend (1) | trend_index |
| Interaction (1) | snap_event_inter |

**Total predictor columns available: 50**

The baseline and traditional models (Naive, ES, ARIMA, SARIMA) use only the raw `sales_qty` time series. These engineered features are only relevant for the ML models.

In the modeling step, features with zero importance scores will be identified and removed, and a cross-validation search will determine the optimal number of features to retain for each department.


In [13]:
print("All columns in final modeling dataset:")
for col in daily_df.columns.tolist():
    print(" -", col)
print(f"\nTotal columns: {len(daily_df.columns)}")


All columns in final modeling dataset:
 - date
 - dept_id
 - weekday
 - month
 - year
 - snap_CA
 - is_event
 - event_count
 - is_cultural_event
 - is_national_event
 - is_religious_event
 - is_sport_event
 - sales_qty
 - sell_price
 - lag_1
 - lag_2
 - lag_3
 - lag_7
 - lag_14
 - lag_21
 - lag_28
 - lag_42
 - lag_56
 - lag_84
 - rolling_mean_7
 - rolling_mean_14
 - rolling_mean_28
 - rolling_mean_42
 - rolling_mean_56
 - rolling_std_7
 - rolling_std_14
 - rolling_std_28
 - rolling_std_42
 - rolling_std_56
 - ewma_7
 - ewma_14
 - ewma_28
 - rolling_max_7
 - rolling_min_7
 - rolling_max_28
 - rolling_min_28
 - day_of_week
 - week_of_year
 - is_weekend
 - day_of_month
 - is_month_end
 - is_quarter_end
 - price_change_7
 - price_change_28
 - sell_price_lag_7
 - sell_price_lag_28
 - trend_index
 - snap_event_inter

Total columns: 53


### **Save Dataset for Modeling**

In [ ]:
output_path = os.path.join(PROCESSED_DIR, "modeling_dataset.csv")
daily_df.to_csv(output_path, index=False)
print("Saved:", output_path)

Saved: /Users/juliuslaurenmarasigan/Documents/FINAL PGDAIML CAPSTONE/processed_data/modeling_dataset.csv
